# Домашнее задание № 10. Генерация текста

### Задание 1 (8 баллов).


Возьмите Russian split из вот этого датасета - https://huggingface.co/datasets/CohereLabs/aya_collection_language_split
Он все равно очень большой, поэтому отфильтруйте его до какого-то небольшого рабочего подмножества (например, до 2000 примеров длиной меньше 300 токенов).
Найдите какую-нибудь языковую модель на huggingface, которая не была дообучена на инструкциях (base). Дообучите ее на получившемся датасете.
Возьмите любую задачу из Russian Superglue (например, вот эту - https://russiansuperglue.com/tasks/task_info/MuSeRC) и создайте из нее небольшой оценочный датасет. Формат должен подходить под получившуюся модель, поэтому если в тексте есть отдельно text и question, то вам понадобится их соединить в один промпт. Сделайте предсказания изначальной моделью и дообученой. Посчитайте какую-нибудь метрику качества или даже несколько (например, точное совпадение с правильными ответом + bleu score). Справляется ли дообученная модель лучше? Проанализируйте несколько предсказаний отдельно.
Вы можете использовать модель любого размера и любую технику дообучения (полное дообучение, LoRA или QLoRA)


(*Это задание сложнее предыдущих, поэтому не стесняйтесь задавать вопросы в чате или лично)


### Задание 2 (2 балла)
Два дополнительных балла можно получить если размер модели больше 3B.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
!pip install trl peft bitsandbytes accelerate datasets transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [ ]:
import transformers

In [1]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
import torch

# 1. Загружаем датасет (только русский)
dataset = load_dataset("CohereLabs/aya_collection_language_split", "russian", split="train")

# 2. Фильтруем по длине токенов (150 токенов)
tokenizer = AutoTokenizer.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def count_tokens(example):
    full_text = f"{example['inputs']}\nОтвет: {example['targets']}"
    return len(tokenizer.encode(full_text, truncation=True, max_length=150)) < 150

# Применяем фильтр (может быть долго, но один раз)
filtered = dataset.filter(count_tokens)
# Берём 200 примеров
small_dataset = filtered.select(range(min(200, len(filtered))))

# 3. Форматируем в тексты
def format_example(example):
    return {"text": f"{example['inputs']}\nОтвет: {example['targets']}"}

formatted = small_dataset.map(format_example)

# 4. Загружаем модель (без квантования, на GPU если есть)
model = AutoModelForCausalLM.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 5. Токенизация с обрезкой до 256 токенов
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length")

tokenized_dataset = formatted.map(tokenize_function, batched=True, remove_columns=["text"])

# 6. Data collator для LM
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 7. Аргументы обучения
training_args = TrainingArguments(
    output_dir="./rugpt3-small-aya",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    learning_rate=5e-5,
    fp16=True,  # если GPU поддерживает
    logging_steps=10,
    save_steps=100,
    save_total_limit=1,
    report_to="none",
)

# 8. Trainer (без аргумента tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# 9. Обучение
trainer.train()

# 10. Сохраняем модель и токенизатор
model.save_pretrained("./rugpt3-small-aya-finetuned")
tokenizer.save_pretrained("./rugpt3-small-aya-finetuned")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

russian/train-00000-of-00002.parquet:   0%|          | 0.00/704M [00:00<?, ?B/s]

russian/train-00001-of-00002.parquet:   0%|          | 0.00/739M [00:00<?, ?B/s]

russian/validation-00000-of-00001.parque(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.486894
20,3.230469


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./rugpt3-small-aya-finetuned/tokenizer_config.json',
 './rugpt3-small-aya-finetuned/tokenizer.json')

In [ ]:
!pip install evaluate

In [7]:
import torch
import re
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score
import evaluate

# ------------------------------------------------------------
# 1. Загрузка моделей
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

base_model_name = "ai-forever/rugpt3small_based_on_gpt2"
finetuned_model_path = "./rugpt3-small-aya-finetuned"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(base_model_name).to(device)
base_model.eval()

finetuned_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_path)
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token
finetuned_model = AutoModelForCausalLM.from_pretrained(finetuned_model_path).to(device)
finetuned_model.eval()

# ------------------------------------------------------------
# 2. Загрузка MuSeRC (исправлено - используем DragonLLM)
# ------------------------------------------------------------
muserc = load_dataset("DragonLLM/russian_super_glue", "muserc", split="validation")
eval_size = 50
eval_data = muserc.select(range(eval_size))

# ------------------------------------------------------------
# 3. Формирование промптов
# ------------------------------------------------------------
def make_prompt(example):
    prompt = f"Текст: {example['paragraph']}\n"
    prompt += f"Вопрос: {example['question']}\n"
    prompt += f"Предполагаемый ответ: {example['answer']}\n"
    prompt += "Ответ:"
    return prompt

prompts = []
true_answers_text = []

for ex in eval_data:
    prompts.append(make_prompt(ex))
    # label: 1 = True (верный ответ), 0 = False
    true_answers_text.append("Да" if ex['label'] == 1 else "Нет")

# ------------------------------------------------------------
# 4. Функция генерации
# ------------------------------------------------------------
def generate_answer(model, tokenizer, prompt, max_new_tokens=5):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Ответ:" in generated:
        answer_part = generated.split("Ответ:")[-1].strip().lower()
    else:
        answer_part = generated.strip().lower()
    if "да" in answer_part and "нет" not in answer_part:
        return "Да"
    elif "нет" in answer_part:
        return "Нет"
    else:
        return "Неизвестно"

# ------------------------------------------------------------
# 5. Предсказания
# ------------------------------------------------------------
base_preds = []
finetuned_preds = []

print("Base model predictions...")
for i, p in enumerate(prompts):
    base_preds.append(generate_answer(base_model, base_tokenizer, p))
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(prompts)}")

print("Finetuned model predictions...")
for i, p in enumerate(prompts):
    finetuned_preds.append(generate_answer(finetuned_model, finetuned_tokenizer, p))
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(prompts)}")

# ------------------------------------------------------------
# 6. Метрики
# ------------------------------------------------------------
base_acc = accuracy_score(true_answers_text, base_preds)
finetuned_acc = accuracy_score(true_answers_text, finetuned_preds)

bleu = evaluate.load("bleu")
base_bleu = bleu.compute(predictions=base_preds, references=[[t] for t in true_answers_text])
finetuned_bleu = bleu.compute(predictions=finetuned_preds, references=[[t] for t in true_answers_text])

print("\n===== RESULTS =====")
print(f"Base model accuracy: {base_acc:.3f}")
print(f"Finetuned model accuracy: {finetuned_acc:.3f}")
print(f"Base model BLEU: {base_bleu['bleu']:.3f}")
print(f"Finetuned model BLEU: {finetuned_bleu['bleu']:.3f}")

# ------------------------------------------------------------
# 7. Анализ примеров
# ------------------------------------------------------------
print("\n===== SAMPLE ANALYSIS =====")
for i in range(min(5, len(prompts))):
    print(f"\nExample {i+1}:")
    print(f"Prompt (first 200 chars): {prompts[i][:200]}...")
    print(f"True answer: {true_answers_text[i]}")
    print(f"Base model: {base_preds[i]}")
    print(f"Finetuned model: {finetuned_preds[i]}")

Using device: cpu


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/7614 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/11950 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2235 [00:00<?, ? examples/s]

Base model predictions...
  10/50
  20/50
  30/50
  40/50
  50/50
Finetuned model predictions...
  10/50
  20/50
  30/50
  40/50
  50/50



===== RESULTS =====
Base model accuracy: 0.020
Finetuned model accuracy: 0.060
Base model BLEU: 0.000
Finetuned model BLEU: 0.000

===== SAMPLE ANALYSIS =====

Example 1:
Prompt (first 200 chars): Текст: (1) Самый первый «остров» Архипелага возник в 1923 году на месте Соловецкого монастыря. (2) Затем появились ТОНы — тюрьмы особого назначения и этапы. (3) Люди попадали на Архипелаг разными спос...
True answer: Да
Base model: Неизвестно
Finetuned model: Неизвестно

Example 2:
Prompt (first 200 chars): Текст: (1) Самый первый «остров» Архипелага возник в 1923 году на месте Соловецкого монастыря. (2) Затем появились ТОНы — тюрьмы особого назначения и этапы. (3) Люди попадали на Архипелаг разными спос...
True answer: Нет
Base model: Неизвестно
Finetuned model: Неизвестно

Example 3:
Prompt (first 200 chars): Текст: (1) Самый первый «остров» Архипелага возник в 1923 году на месте Соловецкого монастыря. (2) Затем появились ТОНы — тюрьмы особого назначения и этапы. (3) Люди попадали на Архип